In [1]:
import numpy as np
from collections import Counter

In [2]:
corpus = [
    "play",
    "playing",
    "played"
]

print("Corpus:")
for word in corpus:
    print(word)

Corpus:
play
playing
played


In [3]:
word_tokens = {}

for word in corpus:
    tokens = list(word)
    word_tokens[word] = tokens

print("Initial tokens:\n")

for word, tokens in word_tokens.items():
    print(word, "->", tokens)

Initial tokens:

play -> ['p', 'l', 'a', 'y']
playing -> ['p', 'l', 'a', 'y', 'i', 'n', 'g']
played -> ['p', 'l', 'a', 'y', 'e', 'd']


In [4]:
def get_token_counts(word_tokens):
    token_counts = Counter()

    for tokens in word_tokens.values():
        for token in tokens:
            token_counts[token] += 1

    return token_counts


token_counts = get_token_counts(word_tokens)

print("Token frequencies:\n")

for token, count in token_counts.items():
    print(token, ":", count)

Token frequencies:

p : 3
l : 3
a : 3
y : 3
i : 1
n : 1
g : 1
e : 1
d : 1


In [5]:
def get_pair_counts(word_tokens):
    pair_counts = Counter()

    for tokens in word_tokens.values():

        for i in range(len(tokens) - 1):

            pair = (tokens[i], tokens[i + 1])
            pair_counts[pair] += 1

    return pair_counts


pair_counts = get_pair_counts(word_tokens)

print("Pair frequencies:\n")

for pair, count in pair_counts.items():
    print(pair, ":", count)

Pair frequencies:

('p', 'l') : 3
('l', 'a') : 3
('a', 'y') : 3
('y', 'i') : 1
('i', 'n') : 1
('n', 'g') : 1
('y', 'e') : 1
('e', 'd') : 1


In [6]:
def calculate_scores(token_counts, pair_counts):

    pair_scores = {}

    for pair, pair_count in pair_counts.items():

        first_token = pair[0]
        second_token = pair[1]

        score = pair_count / (
            token_counts[first_token] *
            token_counts[second_token]
        )

        pair_scores[pair] = score

    return pair_scores


pair_scores = calculate_scores(
    token_counts,
    pair_counts
)

print("WordPiece scores:\n")

for pair, score in pair_scores.items():
    print(pair, ":", round(score, 4))

WordPiece scores:

('p', 'l') : 0.3333
('l', 'a') : 0.3333
('a', 'y') : 0.3333
('y', 'i') : 0.3333
('i', 'n') : 1.0
('n', 'g') : 1.0
('y', 'e') : 0.3333
('e', 'd') : 1.0


In [7]:
best_pair = max(
    pair_scores,
    key=pair_scores.get
)

best_score = pair_scores[best_pair]

print("Best pair:", best_pair)
print("Best score:", round(best_score, 4))

Best pair: ('i', 'n')
Best score: 1.0


In [11]:
def merge_pair(word_tokens, pair):

    new_word_tokens = {}

    for word, tokens in word_tokens.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):

                merged_token = tokens[i] + tokens[i + 1]

                new_tokens.append(merged_token)

                i += 2

            else:

                new_tokens.append(tokens[i])

                i += 1

        new_word_tokens[word] = new_tokens

    return new_word_tokens

In [12]:
vocab = {
    "play",
    "##ing",
    "##ed"
}

print("Vocabulary:")

for token in vocab:
    print(token)

Vocabulary:
##ing
play
##ed


In [13]:
def wordpiece_tokenize(word):

    tokens = []
    start = 0

    while start < len(word):

        found = None

        # Try the longest possible token
        for end in range(len(word), start, -1):

            piece = word[start:end]

            if start > 0:
                piece = "##" + piece

            if piece in vocab:
                found = piece
                next_start = end
                break

        if found is None:
            return ["[UNK]"]

        tokens.append(found)
        start = next_start

    return tokens

In [14]:
print("WordPiece representation:\n")

for word in corpus:
    print(word, "->", wordpiece_tokenize(word))

WordPiece representation:

play -> ['play']
playing -> ['play', '##ing']
played -> ['play', '##ed']


In [ ]:
vocab = {
    "[PAD]": 0,
    "[UNK]": 1,
    "[CLS]": 2,
    "[SEP]": 3,
    "play": 4,
    "##ing": 5,
    "##ed": 6
}

print("Vocabulary:")

for token, token_id in vocab.items():
    print(token, "->", token_id)